# Google Play Review Ingestion and Database Pipeline

In the previous step, I finished the Google Play review database schema design.

In this notebook, I connect that schema to a real ingestion workflow. The main goal is to test whether the pipeline can collect Google Play reviews, process the records, insert them into a SQLite database, handle duplicate reviews, keep ingestion run information, preserve quality flags, and keep raw and cleaned review text linked.

I start with a small controlled batch first, because this is easier to check and debug before scaling up.

In [1]:
!pip install google-play-scraper pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.2 MB/s eta 0:00:00


## 1. Setup

I first import the packages and create folders for the database and output files.

For this first implementation, I use SQLite because it is simple, local, and enough for testing the end-to-end ingestion flow.

In [2]:
import os
import re
import json
import sqlite3
import hashlib
import pandas as pd

from datetime import datetime, timezone
from google_play_scraper import reviews, Sort

In [3]:
BASE_DIR = "/content/google_play_ingestion_database_pipeline"

DB_DIR = os.path.join(BASE_DIR, "database")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs", "run5_ingestion_database_pipeline")

os.makedirs(DB_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

DB_PATH = os.path.join(DB_DIR, "google_play_reviews.sqlite")

print("Base folder:", BASE_DIR)
print("Database path:", DB_PATH)
print("Output folder:", OUTPUT_DIR)

Base folder: /content/google_play_ingestion_database_pipeline
Database path: /content/google_play_ingestion_database_pipeline/database/google_play_reviews.sqlite
Output folder: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline


## 2. Controlled App Batch

John suggested starting with a smaller controlled batch first.

For this run, I use three high-volume apps that were already used in the earlier Google Play validation work. The collection setting is fixed to US / English / newest reviews, so the run is easier to compare across repeated tests.

In [4]:
APP_TARGETS = [
    {
        "app_name": "YouTube",
        "app_id": "com.google.android.youtube",
        "source_platform": "google_play",
        "country": "us",
        "language": "en"
    },
    {
        "app_name": "TikTok",
        "app_id": "com.zhiliaoapp.musically",
        "source_platform": "google_play",
        "country": "us",
        "language": "en"
    },
    {
        "app_name": "Spotify",
        "app_id": "com.spotify.music",
        "source_platform": "google_play",
        "country": "us",
        "language": "en"
    }
]

COUNT_PER_APP = 100
SORT_ORDER = Sort.NEWEST

app_targets_df = pd.DataFrame(APP_TARGETS)
display(app_targets_df)

,app_name,app_id,source_platform,country,language
0,YouTube,com.google.android.youtube,google_play,us,en
1,TikTok,com.zhiliaoapp.musically,google_play,us,en
2,Spotify,com.spotify.music,google_play,us,en


## 3. Database Schema

This database uses the same main structure from the schema design step:

- `app_sources`: app-level source metadata
- `ingestion_runs`: one row for each pipeline run
- `ingestion_run_targets`: app-level result for each run
- `reviews`: review-level metadata
- `review_texts`: raw and cleaned review text
- `review_quality_flags`: quality flags created during ingestion

The duplicate handling is based on the same app source and the source review id. This keeps the same review from being inserted twice during repeated runs.

In [5]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

schema_sql = """
PRAGMA foreign_keys = ON;

CREATE TABLE IF NOT EXISTS app_sources (
    app_source_id INTEGER PRIMARY KEY AUTOINCREMENT,
    source_platform TEXT NOT NULL,
    app_id TEXT NOT NULL,
    app_name TEXT NOT NULL,
    country TEXT NOT NULL,
    language TEXT NOT NULL,
    created_at TEXT NOT NULL,
    UNIQUE(source_platform, app_id, country, language)
);

CREATE TABLE IF NOT EXISTS ingestion_runs (
    run_id TEXT PRIMARY KEY,
    source_platform TEXT NOT NULL,
    run_started_at TEXT NOT NULL,
    run_finished_at TEXT,
    run_status TEXT NOT NULL,
    run_type TEXT NOT NULL,
    scraper_package TEXT,
    sort_order TEXT,
    requested_count_per_app INTEGER,
    notes TEXT
);

CREATE TABLE IF NOT EXISTS ingestion_run_targets (
    run_target_id INTEGER PRIMARY KEY AUTOINCREMENT,
    run_id TEXT NOT NULL,
    app_source_id INTEGER NOT NULL,
    requested_count INTEGER,
    fetched_count INTEGER,
    inserted_new_count INTEGER,
    duplicate_existing_count INTEGER,
    failed_count INTEGER,
    min_review_created_at TEXT,
    max_review_created_at TEXT,
    error_message TEXT,
    created_at TEXT NOT NULL,
    FOREIGN KEY(run_id) REFERENCES ingestion_runs(run_id),
    FOREIGN KEY(app_source_id) REFERENCES app_sources(app_source_id)
);

CREATE TABLE IF NOT EXISTS reviews (
    review_key TEXT PRIMARY KEY,
    app_source_id INTEGER NOT NULL,
    source_review_id TEXT NOT NULL,
    user_name TEXT,
    rating INTEGER,
    thumbs_up_count INTEGER,
    review_created_at TEXT,
    app_version TEXT,
    developer_reply_content TEXT,
    developer_replied_at TEXT,
    raw_response_json TEXT,
    first_seen_run_id TEXT NOT NULL,
    last_seen_run_id TEXT NOT NULL,
    created_at TEXT NOT NULL,
    updated_at TEXT NOT NULL,
    FOREIGN KEY(app_source_id) REFERENCES app_sources(app_source_id),
    FOREIGN KEY(first_seen_run_id) REFERENCES ingestion_runs(run_id),
    FOREIGN KEY(last_seen_run_id) REFERENCES ingestion_runs(run_id),
    UNIQUE(app_source_id, source_review_id)
);

CREATE TABLE IF NOT EXISTS review_texts (
    review_key TEXT PRIMARY KEY,
    raw_text TEXT,
    cleaned_text TEXT,
    raw_text_hash TEXT,
    cleaned_text_hash TEXT,
    created_at TEXT NOT NULL,
    updated_at TEXT NOT NULL,
    FOREIGN KEY(review_key) REFERENCES reviews(review_key)
);

CREATE TABLE IF NOT EXISTS review_quality_flags (
    quality_flag_id INTEGER PRIMARY KEY AUTOINCREMENT,
    run_id TEXT NOT NULL,
    review_key TEXT NOT NULL,
    is_missing_review_id INTEGER NOT NULL,
    is_missing_text INTEGER NOT NULL,
    is_short_text INTEGER NOT NULL,
    is_missing_rating INTEGER NOT NULL,
    is_missing_review_date INTEGER NOT NULL,
    is_repeated_content_in_batch INTEGER NOT NULL,
    content_length INTEGER,
    created_at TEXT NOT NULL,
    FOREIGN KEY(run_id) REFERENCES ingestion_runs(run_id),
    FOREIGN KEY(review_key) REFERENCES reviews(review_key),
    UNIQUE(run_id, review_key)
);
"""

cur.executescript(schema_sql)
conn.commit()

schema_path = os.path.join(OUTPUT_DIR, "schema_used_for_run5.sql")
with open(schema_path, "w", encoding="utf-8") as f:
    f.write(schema_sql)

print("Database tables created.")
print("Saved schema copy to:", schema_path)

Database tables created.
Saved schema copy to: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/schema_used_for_run5.sql


## 4. Helper Functions

These helper functions keep the pipeline easier to read.

The cleaning step is intentionally simple for now. I only trim spaces and normalize repeated whitespace. I do not remove too much text because the raw review content should still be preserved.

In [6]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()


def normalize_datetime(value):
    if value is None:
        return None

    if pd.isna(value):
        return None

    if isinstance(value, datetime):
        if value.tzinfo is None:
            return value.replace(tzinfo=timezone.utc).isoformat()
        return value.astimezone(timezone.utc).isoformat()

    return str(value)


def clean_text(text):
    if text is None:
        return None

    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)

    return text


def hash_text(text):
    if text is None:
        return None

    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()


def make_review_key(app_source_id, source_review_id):
    key_text = f"{app_source_id}|{source_review_id}"
    return hashlib.sha256(key_text.encode("utf-8")).hexdigest()


def get_or_create_app_source(conn, app):
    cur = conn.cursor()
    now = utc_now()

    cur.execute("""
        INSERT OR IGNORE INTO app_sources (
            source_platform,
            app_id,
            app_name,
            country,
            language,
            created_at
        )
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        app["source_platform"],
        app["app_id"],
        app["app_name"],
        app["country"],
        app["language"],
        now
    ))

    conn.commit()

    cur.execute("""
        SELECT app_source_id
        FROM app_sources
        WHERE source_platform = ?
          AND app_id = ?
          AND country = ?
          AND language = ?
    """, (
        app["source_platform"],
        app["app_id"],
        app["country"],
        app["language"]
    ))

    return cur.fetchone()[0]

## 5. Ingestion Function

This function runs the actual ingestion workflow.

For each app, it does these steps:

1. fetch reviews from Google Play
2. process the raw records
3. insert new reviews into the database
4. update existing reviews if they are already in the database
5. store raw and cleaned text
6. create quality flags
7. save app-level run summary

In [7]:
def run_google_play_ingestion(conn, app_targets, count_per_app, run_type, notes):
    cur = conn.cursor()

    run_id = "run5_" + datetime.now().strftime("%Y%m%d_%H%M%S")
    run_started_at = utc_now()

    cur.execute("""
        INSERT INTO ingestion_runs (
            run_id,
            source_platform,
            run_started_at,
            run_status,
            run_type,
            scraper_package,
            sort_order,
            requested_count_per_app,
            notes
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        run_id,
        "google_play",
        run_started_at,
        "running",
        run_type,
        "google-play-scraper",
        "newest",
        count_per_app,
        notes
    ))

    conn.commit()

    print("Started run:", run_id)

    all_processed_records = []
    target_summaries = []

    for app in app_targets:
        print("\nCollecting:", app["app_name"])

        app_source_id = get_or_create_app_source(conn, app)
        fetched_count = 0
        inserted_new_count = 0
        duplicate_existing_count = 0
        failed_count = 0
        error_message = None

        try:
            raw_reviews, continuation_token = reviews(
                app["app_id"],
                lang=app["language"],
                country=app["country"],
                sort=SORT_ORDER,
                count=count_per_app
            )

            fetched_count = len(raw_reviews)
            processed_records = []

            for raw in raw_reviews:
                original_review_id = raw.get("reviewId")
                is_missing_review_id = 1 if original_review_id is None or str(original_review_id).strip() == "" else 0

                raw_text = raw.get("content")
                cleaned_text = clean_text(raw_text)

                raw_text_hash = hash_text(raw_text)
                cleaned_text_hash = hash_text(cleaned_text)

                if is_missing_review_id == 1:
                    fallback_text = f"{app_source_id}|{raw_text_hash}|{raw.get('at')}|{raw.get('score')}"
                    source_review_id = "missing_review_id_" + hash_text(fallback_text)[:16]
                else:
                    source_review_id = str(original_review_id)

                review_key = make_review_key(app_source_id, source_review_id)

                record = {
                    "run_id": run_id,
                    "app_source_id": app_source_id,
                    "app_name": app["app_name"],
                    "app_id": app["app_id"],
                    "country": app["country"],
                    "language": app["language"],
                    "review_key": review_key,
                    "source_review_id": source_review_id,
                    "is_missing_review_id": is_missing_review_id,
                    "user_name": raw.get("userName"),
                    "rating": raw.get("score"),
                    "thumbs_up_count": raw.get("thumbsUpCount"),
                    "review_created_at": normalize_datetime(raw.get("at")),
                    "app_version": raw.get("reviewCreatedVersion"),
                    "developer_reply_content": raw.get("replyContent"),
                    "developer_replied_at": normalize_datetime(raw.get("repliedAt")),
                    "raw_text": raw_text,
                    "cleaned_text": cleaned_text,
                    "raw_text_hash": raw_text_hash,
                    "cleaned_text_hash": cleaned_text_hash,
                    "raw_response_json": json.dumps(raw, default=str, ensure_ascii=False)
                }

                processed_records.append(record)

            non_missing_hashes = [
                r["cleaned_text_hash"]
                for r in processed_records
                if r["cleaned_text_hash"] is not None
            ]

            hash_counts = pd.Series(non_missing_hashes).value_counts()
            repeated_content_hashes = set(hash_counts[hash_counts > 1].index)

            for record in processed_records:
                now = utc_now()

                cur.execute("""
                    SELECT review_key
                    FROM reviews
                    WHERE review_key = ?
                """, (record["review_key"],))

                existed_before = cur.fetchone() is not None

                if existed_before:
                    duplicate_existing_count += 1
                else:
                    inserted_new_count += 1

                cur.execute("""
                    INSERT INTO reviews (
                        review_key,
                        app_source_id,
                        source_review_id,
                        user_name,
                        rating,
                        thumbs_up_count,
                        review_created_at,
                        app_version,
                        developer_reply_content,
                        developer_replied_at,
                        raw_response_json,
                        first_seen_run_id,
                        last_seen_run_id,
                        created_at,
                        updated_at
                    )
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                    ON CONFLICT(review_key) DO UPDATE SET
                        user_name = excluded.user_name,
                        rating = excluded.rating,
                        thumbs_up_count = excluded.thumbs_up_count,
                        review_created_at = excluded.review_created_at,
                        app_version = excluded.app_version,
                        developer_reply_content = excluded.developer_reply_content,
                        developer_replied_at = excluded.developer_replied_at,
                        raw_response_json = excluded.raw_response_json,
                        last_seen_run_id = excluded.last_seen_run_id,
                        updated_at = excluded.updated_at
                """, (
                    record["review_key"],
                    record["app_source_id"],
                    record["source_review_id"],
                    record["user_name"],
                    record["rating"],
                    record["thumbs_up_count"],
                    record["review_created_at"],
                    record["app_version"],
                    record["developer_reply_content"],
                    record["developer_replied_at"],
                    record["raw_response_json"],
                    run_id,
                    run_id,
                    now,
                    now
                ))

                cur.execute("""
                    INSERT INTO review_texts (
                        review_key,
                        raw_text,
                        cleaned_text,
                        raw_text_hash,
                        cleaned_text_hash,
                        created_at,
                        updated_at
                    )
                    VALUES (?, ?, ?, ?, ?, ?, ?)
                    ON CONFLICT(review_key) DO UPDATE SET
                        raw_text = excluded.raw_text,
                        cleaned_text = excluded.cleaned_text,
                        raw_text_hash = excluded.raw_text_hash,
                        cleaned_text_hash = excluded.cleaned_text_hash,
                        updated_at = excluded.updated_at
                """, (
                    record["review_key"],
                    record["raw_text"],
                    record["cleaned_text"],
                    record["raw_text_hash"],
                    record["cleaned_text_hash"],
                    now,
                    now
                ))

                text = record["cleaned_text"]
                content_length = len(text) if text else 0

                is_missing_text = 1 if text is None or text == "" else 0
                is_short_text = 1 if content_length > 0 and content_length < 5 else 0
                is_missing_rating = 1 if record["rating"] is None else 0
                is_missing_review_date = 1 if record["review_created_at"] is None else 0
                is_repeated_content_in_batch = 1 if record["cleaned_text_hash"] in repeated_content_hashes else 0

                cur.execute("""
                    INSERT OR IGNORE INTO review_quality_flags (
                        run_id,
                        review_key,
                        is_missing_review_id,
                        is_missing_text,
                        is_short_text,
                        is_missing_rating,
                        is_missing_review_date,
                        is_repeated_content_in_batch,
                        content_length,
                        created_at
                    )
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """, (
                    run_id,
                    record["review_key"],
                    record["is_missing_review_id"],
                    is_missing_text,
                    is_short_text,
                    is_missing_rating,
                    is_missing_review_date,
                    is_repeated_content_in_batch,
                    content_length,
                    now
                ))

            conn.commit()

            all_processed_records.extend(processed_records)

            review_dates = [
                r["review_created_at"]
                for r in processed_records
                if r["review_created_at"] is not None
            ]

            min_review_created_at = min(review_dates) if len(review_dates) > 0 else None
            max_review_created_at = max(review_dates) if len(review_dates) > 0 else None

            print("Fetched:", fetched_count)
            print("Inserted new:", inserted_new_count)
            print("Duplicates:", duplicate_existing_count)

        except Exception as e:
            failed_count = 1
            error_message = str(e)
            min_review_created_at = None
            max_review_created_at = None

            print("Failed:", app["app_name"])
            print("Error:", error_message)

        target_created_at = utc_now()

        cur.execute("""
            INSERT INTO ingestion_run_targets (
                run_id,
                app_source_id,
                requested_count,
                fetched_count,
                inserted_new_count,
                duplicate_existing_count,
                failed_count,
                min_review_created_at,
                max_review_created_at,
                error_message,
                created_at
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            run_id,
            app_source_id,
            count_per_app,
            fetched_count,
            inserted_new_count,
            duplicate_existing_count,
            failed_count,
            min_review_created_at,
            max_review_created_at,
            error_message,
            target_created_at
        ))

        target_summaries.append({
            "run_id": run_id,
            "app_name": app["app_name"],
            "app_id": app["app_id"],
            "requested_count": count_per_app,
            "fetched_count": fetched_count,
            "inserted_new_count": inserted_new_count,
            "duplicate_existing_count": duplicate_existing_count,
            "failed_count": failed_count,
            "min_review_created_at": min_review_created_at,
            "max_review_created_at": max_review_created_at,
            "error_message": error_message
        })

        conn.commit()

    run_finished_at = utc_now()

    cur.execute("""
        UPDATE ingestion_runs
        SET run_finished_at = ?,
            run_status = ?
        WHERE run_id = ?
    """, (
        run_finished_at,
        "completed",
        run_id
    ))

    conn.commit()

    processed_df = pd.DataFrame(all_processed_records)
    target_summary_df = pd.DataFrame(target_summaries)

    print("\nCompleted run:", run_id)

    return run_id, processed_df, target_summary_df

## 6. First Controlled Run

This is the first real end-to-end ingestion run.

I expect this run to insert mostly new reviews because the database is empty at the beginning.

In [8]:
run1_id, run1_records_df, run1_target_summary_df = run_google_play_ingestion(
    conn=conn,
    app_targets=APP_TARGETS,
    count_per_app=COUNT_PER_APP,
    run_type="controlled_batch_initial",
    notes="First controlled ingestion run for database implementation test."
)

display(run1_target_summary_df)

Started run: run5_20260702_050046

Collecting: YouTube
Fetched: 100
Inserted new: 100
Duplicates: 0

Collecting: TikTok
Fetched: 100
Inserted new: 100
Duplicates: 0

Collecting: Spotify
Fetched: 100
Inserted new: 100
Duplicates: 0

Completed run: run5_20260702_050046


,run_id,app_name,app_id,requested_count,fetched_count,inserted_new_count,duplicate_existing_count,failed_count,min_review_created_at,max_review_created_at,error_message
0,run5_20260702_050046,YouTube,com.google.android.youtube,100,100,100,0,0,2026-07-01T04:02:55+00:00,2026-07-01T05:00:41+00:00,None
1,run5_20260702_050046,TikTok,com.zhiliaoapp.musically,100,100,100,0,0,2026-07-01T00:23:21+00:00,2026-07-01T04:55:51+00:00,None
2,run5_20260702_050046,Spotify,com.spotify.music,100,100,100,0,0,2026-07-01T00:44:56+00:00,2026-07-01T05:00:36+00:00,None


## 7. Duplicate Handling Test

Now I run the same collection again.

If duplicate handling works, the second run should find many existing reviews. Those records should not be inserted again as new database rows. Instead, the pipeline updates the existing rows and changes `last_seen_run_id`.

In [9]:
run2_id, run2_records_df, run2_target_summary_df = run_google_play_ingestion(
    conn=conn,
    app_targets=APP_TARGETS,
    count_per_app=COUNT_PER_APP,
    run_type="controlled_batch_duplicate_check",
    notes="Second run using the same targets to test duplicate handling."
)

display(run2_target_summary_df)

Started run: run5_20260702_050059

Collecting: YouTube
Fetched: 100
Inserted new: 0
Duplicates: 100

Collecting: TikTok
Fetched: 100
Inserted new: 0
Duplicates: 100

Collecting: Spotify
Fetched: 100
Inserted new: 0
Duplicates: 100

Completed run: run5_20260702_050059


,run_id,app_name,app_id,requested_count,fetched_count,inserted_new_count,duplicate_existing_count,failed_count,min_review_created_at,max_review_created_at,error_message
0,run5_20260702_050059,YouTube,com.google.android.youtube,100,100,0,100,0,2026-07-01T04:02:55+00:00,2026-07-01T05:00:41+00:00,None
1,run5_20260702_050059,TikTok,com.zhiliaoapp.musically,100,100,0,100,0,2026-07-01T00:23:21+00:00,2026-07-01T04:55:51+00:00,None
2,run5_20260702_050059,Spotify,com.spotify.music,100,100,0,100,0,2026-07-01T00:44:56+00:00,2026-07-01T05:00:36+00:00,None


## 8. Run-Level Summary

This table checks whether each ingestion run was recorded correctly.

In [11]:
ingestion_run_summary = pd.read_sql_query("""
    SELECT
        run_id,
        source_platform,
        run_started_at,
        run_finished_at,
        run_status,
        run_type,
        scraper_package,
        sort_order,
        requested_count_per_app,
        notes
    FROM ingestion_runs
    ORDER BY run_started_at;
""", conn)

display(ingestion_run_summary)

ingestion_run_summary_path = os.path.join(OUTPUT_DIR, "ingestion_run_summary.csv")
ingestion_run_summary.to_csv(ingestion_run_summary_path, index=False)

print("Saved:", ingestion_run_summary_path)

,run_id,source_platform,run_started_at,run_finished_at,run_status,run_type,scraper_package,sort_order,requested_count_per_app,notes
0,run5_20260702_050046,google_play,2026-07-02T05:00:46.526983+00:00,2026-07-02T05:00:47.617467+00:00,completed,controlled_batch_initial,google-play-scraper,newest,100,First controlled ingestion run for database im...
1,run5_20260702_050059,google_play,2026-07-02T05:00:59.782587+00:00,2026-07-02T05:01:00.347090+00:00,completed,controlled_batch_duplicate_check,google-play-scraper,newest,100,Second run using the same targets to test dupl...


Saved: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/ingestion_run_summary.csv


## 9. App-Level Ingestion Summary

This table shows the result for each app in each run.

The most important columns here are:

- `fetched_count`
- `inserted_new_count`
- `duplicate_existing_count`

For the second run, I expect the duplicate count to increase because many reviews were already inserted in the first run.

In [12]:
ingestion_target_summary = pd.read_sql_query("""
    SELECT
        irt.run_id,
        ir.run_type,
        a.app_name,
        a.app_id,
        a.country,
        a.language,
        irt.requested_count,
        irt.fetched_count,
        irt.inserted_new_count,
        irt.duplicate_existing_count,
        irt.failed_count,
        irt.min_review_created_at,
        irt.max_review_created_at,
        irt.error_message
    FROM ingestion_run_targets irt
    JOIN ingestion_runs ir
        ON irt.run_id = ir.run_id
    JOIN app_sources a
        ON irt.app_source_id = a.app_source_id
    ORDER BY irt.run_id, a.app_name;
""", conn)

display(ingestion_target_summary)

ingestion_target_summary_path = os.path.join(OUTPUT_DIR, "ingestion_target_summary.csv")
ingestion_target_summary.to_csv(ingestion_target_summary_path, index=False)

print("Saved:", ingestion_target_summary_path)

,run_id,run_type,app_name,app_id,country,language,requested_count,fetched_count,inserted_new_count,duplicate_existing_count,failed_count,min_review_created_at,max_review_created_at,error_message
0,run5_20260702_050046,controlled_batch_initial,Spotify,com.spotify.music,us,en,100,100,100,0,0,2026-07-01T00:44:56+00:00,2026-07-01T05:00:36+00:00,None
1,run5_20260702_050046,controlled_batch_initial,TikTok,com.zhiliaoapp.musically,us,en,100,100,100,0,0,2026-07-01T00:23:21+00:00,2026-07-01T04:55:51+00:00,None
2,run5_20260702_050046,controlled_batch_initial,YouTube,com.google.android.youtube,us,en,100,100,100,0,0,2026-07-01T04:02:55+00:00,2026-07-01T05:00:41+00:00,None
3,run5_20260702_050059,controlled_batch_duplicate_check,Spotify,com.spotify.music,us,en,100,100,0,100,0,2026-07-01T00:44:56+00:00,2026-07-01T05:00:36+00:00,None
4,run5_20260702_050059,controlled_batch_duplicate_check,TikTok,com.zhiliaoapp.musically,us,en,100,100,0,100,0,2026-07-01T00:23:21+00:00,2026-07-01T04:55:51+00:00,None
5,run5_20260702_050059,controlled_batch_duplicate_check,YouTube,com.google.android.youtube,us,en,100,100,0,100,0,2026-07-01T04:02:55+00:00,2026-07-01T05:00:41+00:00,None


Saved: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/ingestion_target_summary.csv


## 10. Database Validation Checks

These checks make sure the database tables are connected correctly.

The two checks I care about most are:

1. there should be no duplicate review rows for the same app source and source review id
2. every review should have a linked row in `review_texts`

In [13]:
validation_checks = []

total_app_sources = pd.read_sql_query("SELECT COUNT(*) AS value FROM app_sources;", conn).loc[0, "value"]
total_runs = pd.read_sql_query("SELECT COUNT(*) AS value FROM ingestion_runs;", conn).loc[0, "value"]
total_reviews = pd.read_sql_query("SELECT COUNT(*) AS value FROM reviews;", conn).loc[0, "value"]
total_review_texts = pd.read_sql_query("SELECT COUNT(*) AS value FROM review_texts;", conn).loc[0, "value"]
total_quality_flags = pd.read_sql_query("SELECT COUNT(*) AS value FROM review_quality_flags;", conn).loc[0, "value"]

duplicate_review_rows = pd.read_sql_query("""
    SELECT COUNT(*) AS value
    FROM (
        SELECT app_source_id, source_review_id, COUNT(*) AS row_count
        FROM reviews
        GROUP BY app_source_id, source_review_id
        HAVING COUNT(*) > 1
    );
""", conn).loc[0, "value"]

reviews_without_text_link = pd.read_sql_query("""
    SELECT COUNT(*) AS value
    FROM reviews r
    LEFT JOIN review_texts t
        ON r.review_key = t.review_key
    WHERE t.review_key IS NULL;
""", conn).loc[0, "value"]

quality_flags_without_review = pd.read_sql_query("""
    SELECT COUNT(*) AS value
    FROM review_quality_flags q
    LEFT JOIN reviews r
        ON q.review_key = r.review_key
    WHERE r.review_key IS NULL;
""", conn).loc[0, "value"]

validation_checks.append({"check_name": "total_app_sources", "result_value": total_app_sources})
validation_checks.append({"check_name": "total_ingestion_runs", "result_value": total_runs})
validation_checks.append({"check_name": "total_reviews", "result_value": total_reviews})
validation_checks.append({"check_name": "total_review_texts", "result_value": total_review_texts})
validation_checks.append({"check_name": "total_quality_flags", "result_value": total_quality_flags})
validation_checks.append({"check_name": "duplicate_review_rows_same_app_source", "result_value": duplicate_review_rows})
validation_checks.append({"check_name": "reviews_without_text_link", "result_value": reviews_without_text_link})
validation_checks.append({"check_name": "quality_flags_without_review", "result_value": quality_flags_without_review})

database_validation_summary = pd.DataFrame(validation_checks)

display(database_validation_summary)

database_validation_summary_path = os.path.join(OUTPUT_DIR, "database_validation_summary.csv")
database_validation_summary.to_csv(database_validation_summary_path, index=False)

print("Saved:", database_validation_summary_path)

,check_name,result_value
0,total_app_sources,3
1,total_ingestion_runs,2
2,total_reviews,300
3,total_review_texts,300
4,total_quality_flags,600
5,duplicate_review_rows_same_app_source,0
6,reviews_without_text_link,0
7,quality_flags_without_review,0


Saved: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/database_validation_summary.csv


## 11. Quality Flag Summary

This checks whether the quality flags are preserved in the database.

The flags are simple for now, but they are useful for checking missing text, very short text, missing ratings, missing review dates, and repeated content within the same collected batch.

In [14]:
quality_flag_summary = pd.read_sql_query("""
    SELECT
        q.run_id,
        ir.run_type,
        COUNT(*) AS total_flag_rows,
        SUM(q.is_missing_review_id) AS missing_review_id_count,
        SUM(q.is_missing_text) AS missing_text_count,
        SUM(q.is_short_text) AS short_text_count,
        SUM(q.is_missing_rating) AS missing_rating_count,
        SUM(q.is_missing_review_date) AS missing_review_date_count,
        SUM(q.is_repeated_content_in_batch) AS repeated_content_in_batch_count,
        AVG(q.content_length) AS avg_content_length
    FROM review_quality_flags q
    JOIN ingestion_runs ir
        ON q.run_id = ir.run_id
    GROUP BY q.run_id, ir.run_type
    ORDER BY q.run_id;
""", conn)

display(quality_flag_summary)

quality_flag_summary_path = os.path.join(OUTPUT_DIR, "quality_flag_summary.csv")
quality_flag_summary.to_csv(quality_flag_summary_path, index=False)

print("Saved:", quality_flag_summary_path)

,run_id,run_type,total_flag_rows,missing_review_id_count,missing_text_count,short_text_count,missing_rating_count,missing_review_date_count,repeated_content_in_batch_count,avg_content_length
0,run5_20260702_050046,controlled_batch_initial,300,0,0,27,0,0,16,81.476667
1,run5_20260702_050059,controlled_batch_duplicate_check,300,0,0,27,0,0,16,81.476667


Saved: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/quality_flag_summary.csv


## 12. Raw and Cleaned Text Link Check

This sample confirms that the raw review text and cleaned review text are stored separately, but still linked through the same `review_key`.

In [15]:
sample_reviews = pd.read_sql_query("""
    SELECT
        a.app_name,
        r.source_review_id,
        r.rating,
        r.thumbs_up_count,
        r.review_created_at,
        r.app_version,
        r.first_seen_run_id,
        r.last_seen_run_id,
        t.raw_text,
        t.cleaned_text,
        t.raw_text_hash,
        t.cleaned_text_hash
    FROM reviews r
    JOIN app_sources a
        ON r.app_source_id = a.app_source_id
    JOIN review_texts t
        ON r.review_key = t.review_key
    ORDER BY r.updated_at DESC
    LIMIT 20;
""", conn)

display(sample_reviews)

sample_reviews_path = os.path.join(OUTPUT_DIR, "sample_inserted_reviews.csv")
sample_reviews.to_csv(sample_reviews_path, index=False)

print("Saved:", sample_reviews_path)

,app_name,source_review_id,rating,thumbs_up_count,review_created_at,app_version,first_seen_run_id,last_seen_run_id,raw_text,cleaned_text,raw_text_hash,cleaned_text_hash
0,Spotify,5a506b8b-63be-43a2-9a6a-9012f6927ac0,4,0,2026-07-01T00:44:56+00:00,9.1.60.1970,run5_20260702_050046,run5_20260702_050059,"absolutely epic,but minus one star because the...","absolutely epic,but minus one star because the...",82a1556ead812589f891a2070387c72095ab3aa4e79e57...,82a1556ead812589f891a2070387c72095ab3aa4e79e57...
1,Spotify,88fc638b-058e-4fdf-b37b-07ff4ef44ed1,3,0,2026-07-01T00:46:41+00:00,None,run5_20260702_050046,run5_20260702_050059,Decent but annoying adds,Decent but annoying adds,1f6964e862c2bf72216ca9a8eb398dcff870709de6e526...,1f6964e862c2bf72216ca9a8eb398dcff870709de6e526...
2,Spotify,eb54234c-9b0e-4624-b7b2-ae604609abaf,1,1,2026-07-01T00:51:33+00:00,9.1.60.1970,run5_20260702_050046,run5_20260702_050059,The app is run by monsters..hard to let Google...,The app is run by monsters..hard to let Google...,645cdfdebd99180cae879fd2960bcfb7dfd6fe1894ffbc...,645cdfdebd99180cae879fd2960bcfb7dfd6fe1894ffbc...
3,Spotify,763cb716-0b00-4aea-9bd0-27d279dde6cc,1,0,2026-07-01T00:51:55+00:00,9.1.56.574,run5_20260702_050046,run5_20260702_050059,Why tf is spotify donating to AI drone weapon ...,Why tf is spotify donating to AI drone weapon ...,7728939398e6be1fa6d446e976724452e40e7d7e26c84d...,7728939398e6be1fa6d446e976724452e40e7d7e26c84d...
4,Spotify,9dc804cd-8149-454c-aa26-ca9f480d4e25,5,0,2026-07-01T00:52:14+00:00,9.1.60.1970,run5_20260702_050046,run5_20260702_050059,if there is a song you want to listen to Spoti...,if there is a song you want to listen to Spoti...,e1f94c8a91f938f406d861cee37cb4a92f4d1aaa17c9e2...,e1f94c8a91f938f406d861cee37cb4a92f4d1aaa17c9e2...
5,Spotify,5e4a4d79-5dc2-41de-950b-61f67dc4de30,4,0,2026-07-01T00:53:35+00:00,8.9.24.633,run5_20260702_050046,run5_20260702_050059,good,good,770e607624d689265ca6c44884d0807d9b054d23c473c1...,770e607624d689265ca6c44884d0807d9b054d23c473c1...
6,Spotify,14046bed-cca6-4ee2-86b8-dc3feffb9012,4,0,2026-07-01T00:54:20+00:00,9.1.60.1970,run5_20260702_050046,run5_20260702_050059,Spotify always hitting with the best tunes and...,Spotify always hitting with the best tunes and...,06dd7c13e7f1ff4da1851ed0e4bf0141c359388debc69b...,06dd7c13e7f1ff4da1851ed0e4bf0141c359388debc69b...
7,Spotify,88dd9b95-d87e-4216-a4e7-34975f7effb2,2,0,2026-07-01T01:02:26+00:00,None,run5_20260702_050046,run5_20260702_050059,it starts automatically when I get in the car ...,it starts automatically when I get in the car ...,7cb22c5fc82d39828db2a274eea771f5d92ddb17dafd01...,7cb22c5fc82d39828db2a274eea771f5d92ddb17dafd01...
8,Spotify,26d14302-76c0-48f2-a44a-3664d230b008,5,0,2026-07-01T01:02:29+00:00,9.1.60.1970,run5_20260702_050046,run5_20260702_050059,nice 👍👍,nice 👍👍,a863162487c580fd5e00d2b3a1ae32721e5116d5bef71b...,a863162487c580fd5e00d2b3a1ae32721e5116d5bef71b...
9,Spotify,52f21d6f-ec81-4c49-b701-08e854774c1b,1,0,2026-07-01T01:04:40+00:00,None,run5_20260702_050046,run5_20260702_050059,My Rating On This Online Radio Please Avoid. W...,My Rating On This Online Radio Please Avoid. W...,5d8b721f987adad82ad35d10ffa43c2a8462ec5b59e285...,5d8b721f987adad82ad35d10ffa43c2a8462ec5b59e285...


Saved: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/sample_inserted_reviews.csv


## 13. Duplicate Handling Check

This is the main check for duplicate handling.

The second run should not create duplicate review rows. Instead, already-seen reviews should be counted under `duplicate_existing_count`, and the original review row should be updated with the newer `last_seen_run_id`.

In [16]:
duplicate_handling_check = pd.read_sql_query("""
    SELECT
        irt.run_id,
        ir.run_type,
        a.app_name,
        irt.requested_count,
        irt.fetched_count,
        irt.inserted_new_count,
        irt.duplicate_existing_count,
        irt.failed_count,
        irt.min_review_created_at,
        irt.max_review_created_at
    FROM ingestion_run_targets irt
    JOIN ingestion_runs ir
        ON irt.run_id = ir.run_id
    JOIN app_sources a
        ON irt.app_source_id = a.app_source_id
    ORDER BY irt.run_id, a.app_name;
""", conn)

display(duplicate_handling_check)

duplicate_handling_check_path = os.path.join(OUTPUT_DIR, "duplicate_handling_check.csv")
duplicate_handling_check.to_csv(duplicate_handling_check_path, index=False)

print("Saved:", duplicate_handling_check_path)

,run_id,run_type,app_name,requested_count,fetched_count,inserted_new_count,duplicate_existing_count,failed_count,min_review_created_at,max_review_created_at
0,run5_20260702_050046,controlled_batch_initial,Spotify,100,100,100,0,0,2026-07-01T00:44:56+00:00,2026-07-01T05:00:36+00:00
1,run5_20260702_050046,controlled_batch_initial,TikTok,100,100,100,0,0,2026-07-01T00:23:21+00:00,2026-07-01T04:55:51+00:00
2,run5_20260702_050046,controlled_batch_initial,YouTube,100,100,100,0,0,2026-07-01T04:02:55+00:00,2026-07-01T05:00:41+00:00
3,run5_20260702_050059,controlled_batch_duplicate_check,Spotify,100,100,0,100,0,2026-07-01T00:44:56+00:00,2026-07-01T05:00:36+00:00
4,run5_20260702_050059,controlled_batch_duplicate_check,TikTok,100,100,0,100,0,2026-07-01T00:23:21+00:00,2026-07-01T04:55:51+00:00
5,run5_20260702_050059,controlled_batch_duplicate_check,YouTube,100,100,0,100,0,2026-07-01T04:02:55+00:00,2026-07-01T05:00:41+00:00


Saved: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/duplicate_handling_check.csv


## 14. Table Row Counts

This is a quick final check of how many rows were created in each table.

In [17]:
table_names = [
    "app_sources",
    "ingestion_runs",
    "ingestion_run_targets",
    "reviews",
    "review_texts",
    "review_quality_flags"
]

table_counts = []

for table in table_names:
    count = pd.read_sql_query(f"SELECT COUNT(*) AS count FROM {table};", conn).loc[0, "count"]
    table_counts.append({
        "table_name": table,
        "row_count": count
    })

table_counts_df = pd.DataFrame(table_counts)
display(table_counts_df)

table_counts_path = os.path.join(OUTPUT_DIR, "table_counts.csv")
table_counts_df.to_csv(table_counts_path, index=False)

print("Saved:", table_counts_path)

,table_name,row_count
0,app_sources,3
1,ingestion_runs,2
2,ingestion_run_targets,6
3,reviews,300
4,review_texts,300
5,review_quality_flags,600


Saved: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/table_counts.csv


## 15. Save Full Review-Level Output

This file is only for checking. The main storage is still the SQLite database.

In [18]:
review_level_output = pd.read_sql_query("""
    SELECT
        a.source_platform,
        a.app_name,
        a.app_id,
        a.country,
        a.language,
        r.source_review_id,
        r.rating,
        r.thumbs_up_count,
        r.review_created_at,
        r.app_version,
        r.first_seen_run_id,
        r.last_seen_run_id,
        t.raw_text,
        t.cleaned_text,
        t.raw_text_hash,
        t.cleaned_text_hash
    FROM reviews r
    JOIN app_sources a
        ON r.app_source_id = a.app_source_id
    JOIN review_texts t
        ON r.review_key = t.review_key
    ORDER BY a.app_name, r.review_created_at DESC;
""", conn)

review_level_output_path = os.path.join(OUTPUT_DIR, "review_level_database_export.csv")
review_level_output.to_csv(review_level_output_path, index=False)

print("Review-level export shape:", review_level_output.shape)
print("Saved:", review_level_output_path)

Review-level export shape: (300, 16)
Saved: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/review_level_database_export.csv


## 16. Create a Zip File for Download

I save the database and output files together so they can be uploaded to GitHub later.

In [19]:
import shutil

zip_base_path = os.path.join(BASE_DIR, "run5_ingestion_database_pipeline_files")
zip_path = shutil.make_archive(zip_base_path, "zip", BASE_DIR)

print("Created zip file:", zip_path)

Created zip file: /content/google_play_ingestion_database_pipeline/run5_ingestion_database_pipeline_files.zip


## Final Conclusion

This notebook connects the Google Play review database schema to a working ingestion pipeline.

The pipeline now completes the basic end-to-end flow:

1. collects Google Play reviews from a controlled app batch
2. creates and connects to a SQLite database
3. records each ingestion run
4. stores app-level ingestion results
5. inserts new review-level records
6. keeps raw and cleaned review text linked through the same review key
7. creates quality flags for basic data quality checks
8. handles duplicate reviews during repeated runs
9. saves validation outputs for checking and GitHub documentation

The first run is used to test new review insertion. The second run is used to test duplicate handling. If the pipeline works correctly, the second run should show higher duplicate counts and should not create duplicate review rows.

This completes the first implementation step from schema design to an actual ingestion and database workflow. The next step is to run the same pipeline across a few different collection times to check stability and whether new reviews can be captured over time.

### Result Summary from This Controlled Test

The first controlled ingestion run successfully collected 300 Google Play reviews across three apps and inserted all 300 records as new database rows.

The second controlled run used the same app targets and collection settings to test duplicate handling. It fetched 300 reviews again, but inserted 0 new rows and identified all 300 records as existing duplicates.

The database validation checks also passed. There were 0 duplicate review rows for the same app source and source review id, 0 reviews without linked text records, and 0 quality flags without linked review records.

This confirms that the basic end-to-end ingestion and database workflow is working. The pipeline can collect reviews, process records, insert new rows, handle duplicates, record ingestion run information, preserve quality flags, and keep raw and cleaned text linked.

The immediate repeated run confirms duplicate handling. The next step is to run the same pipeline at a few later collection times to test run-to-run stability and whether new reviews can be captured over time.